# Data Retrieval Pipeline: VIX, VVIX, and SVXY
**MScFE 690 Capstone -Group 17843**

**Authors:** Edgar NAVA-edgar.nava@grovwm.com & Celestin NYANDWI-celestinnyandwi48@gmail.com

**Purpose:** This notebook downloads historical daily OHLC data from Yahoo Finance for the VIX, VVIX, and SVXY, cleans the datasets, and exports them as CSVs to a local `data/` folder for downstream Markov modeling. The study end date is fixed at September 16, 2026[cite: 4].

### 1. Environment Setup & Dependencies
Ensure all required external libraries are installed for the local environment.

In [1]:
# Install necessary packages (run once)
!pip install -q pandas yfinance

### 2. Global Configuration & Local Directory Setup
Establish the local `data/` directory for saving the final CSVs. Define the global study parameters, including the fixed end date of September 16, 2026, and the required OHLC columns to standardize the outputs.

In [2]:
from pathlib import Path
import pandas as pd
import yfinance as yf

# Create a local 'data' folder in the same directory as this notebook
OUTPUT_FOLDER = Path.cwd() / "data"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# Fixed study end date
END_DATE = pd.Timestamp("2026-09-16")

# Required columns for the final output
OUTPUT_COLUMNS = ["Date", "Open", "High", "Low", "Close"]

print(f"Environment ready. Data will be saved locally to: {OUTPUT_FOLDER}")

Environment ready. Data will be saved locally to: /content/data


### 3. VIX and VVIX Index Retrieval Engine
Define the function to retrieve historical daily OHLC data from the Yahoo Finance API. The engine strips timezone data, drops rows with missing values, rounds price points to six decimal places for precision, and exports the cleaned datasets to the local directory.

In [3]:
# Configuration for VIX and VVIX
INDEX_CONFIG = {
    "VVIX": {
        "ticker": "^VVIX",
        "start_date": pd.Timestamp("2007-01-03"),
        "output_file": "VVIX_data.csv",
    },
    "VIX": {
        "ticker": "^VIX",
        "start_date": pd.Timestamp("2004-03-26"),
        "output_file": "VIX_data.csv",
    },
}

def download_index_data(index_name, ticker_symbol, start_date, end_date, output_file):
    print(f"Downloading {index_name} data...")
    yahoo_end_date = end_date + pd.Timedelta(days=1)
    ticker = yf.Ticker(ticker_symbol)

    data = ticker.history(
        start=start_date.strftime("%Y-%m-%d"),
        end=yahoo_end_date.strftime("%Y-%m-%d"),
        interval="1d",
        auto_adjust=False,
        actions=False,
    )

    if data.empty:
        raise ValueError(f"No historical data returned for {index_name}.")

    if data.index.tz is not None:
        data.index = data.index.tz_localize(None)

    data = data.reset_index()
    data = data.rename(columns={data.columns[0]: "Date"})
    data["Date"] = pd.to_datetime(data["Date"], errors="coerce")

    data = data.loc[(data["Date"] >= start_date) & (data["Date"] <= end_date)].copy()
    data = data.dropna(subset=OUTPUT_COLUMNS)
    data = data.sort_values("Date").reset_index(drop=True)

    data = data[OUTPUT_COLUMNS]
    price_columns = ["Open", "High", "Low", "Close"]
    data[price_columns] = data[price_columns].round(6)
    data["Date"] = data["Date"].dt.strftime("%Y-%m-%d")

    output_path = OUTPUT_FOLDER / output_file
    data.to_csv(output_path, index=False, encoding="utf-8")

    print(f"Saved {index_name} to {output_path} | Observations: {len(data):,}")
    return data

# Execute downloads
historical_data = {}
for index_name, config in INDEX_CONFIG.items():
    historical_data[index_name] = download_index_data(
        index_name, config["ticker"], config["start_date"], END_DATE, config["output_file"]
    )

Saved VVIX to /content/data/VVIX_data.csv | Observations: 4,948
Saved VIX to /content/data/VIX_data.csv | Observations: 5,656


### 4. SVXY ETF Retrieval Engine
Retrieve the historical data for the ProShares Short VIX Short-Term Futures ETF (SVXY) starting from the requested retrieval start date (October 4, 2011). The same cleaning and formatting standards are applied before saving.

In [ ]:
# Configuration for SVXY
SVXY_TICKER = "SVXY"
SVXY_START = pd.Timestamp("2011-10-04")
SVXY_FILE = "SVXY_data.csv"

def download_svxy_data():
    print("Downloading SVXY historical data...")
    yahoo_end_date = END_DATE + pd.Timedelta(days=1)
    ticker = yf.Ticker(SVXY_TICKER)

    data = ticker.history(
        start=SVXY_START.strftime("%Y-%m-%d"),
        end=yahoo_end_date.strftime("%Y-%m-%d"),
        interval="1d",
        auto_adjust=False,
        actions=False,
    )

    if data.empty:
        raise ValueError("No historical data returned for SVXY.")

    if data.index.tz is not None:
        data.index = data.index.tz_localize(None)

    data = data.reset_index()
    data = data.rename(columns={data.columns[0]: "Date"})
    data["Date"] = pd.to_datetime(data["Date"], errors="coerce")

    data = data.loc[(data["Date"] >= SVXY_START) & (data["Date"] <= END_DATE)].copy()
    data = data.dropna(subset=OUTPUT_COLUMNS)
    data = data.sort_values("Date").reset_index(drop=True)

    data = data[OUTPUT_COLUMNS]
    price_columns = ["Open", "High", "Low", "Close"]
    data[price_columns] = data[price_columns].round(6)
    data["Date"] = data["Date"].dt.strftime("%Y-%m-%d")

    output_path = OUTPUT_FOLDER / SVXY_FILE
    data.to_csv(output_path, index=False, encoding="utf-8")

    print(f"Saved SVXY to {output_path} | Observations: {len(data):,}")
    return data

# Execute SVXY download
svxy_data = download_svxy_data()

### 5. Data Validation & Summary Statistics
Compile the downloaded datasets into a summary table to verify that all date ranges align with the Capstone requirements and that the expected number of observations were successfully captured.

In [ ]:
# Compile and display a summary dataframe of all downloaded data
datasets = {**historical_data, "SVXY": svxy_data}

summary = pd.DataFrame([
    {
        "Dataset": name,
        "Observations": len(data),
        "First Date": data["Date"].iloc[0],
        "Last Date": data["Date"].iloc[-1]
    }
    for name, data in datasets.items()
])

print("\nData Retrieval Successfully Completed.")
display(summary)